## OLD

In [2]:
import os
import sys
import math

os.chdir("/workspace/StableVQA")
sys.path.insert(0, "/workspace/StableVQA")

import cv2
import yaml
import torch
import random
import shutil
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from onnx.shape_inference import infer_shapes_path
from onnxruntime.quantization import quant_pre_process

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

VIDEO_FOLDER    = "/workspace/VSdataset2.0"
CONFIG_PATH     = "/workspace/StableVQA/options/stable.yml"
WEIGHTS_PATH    = "/workspace/StableVQA/pretrained_weights/Stable-VQA-M-VS_val_n_dev_v1.0.pth"
NEUFLOW_WEIGHTS = "/workspace/StableVQA/NeuFlow/neuflow_mixed.pth"

NUM_CALIB_VIDS = 200
RANDOM_SEED    = 42

PREP_DIR  = "NEWonnx_static_prep"
CALIB_DIR = "NEWcalib_data"

MEAN = np.array([123.675, 116.28,  103.53], dtype=np.float32)
STD  = np.array([ 58.395,  57.12,  57.375], dtype=np.float32)

TARGET_SIZE = 224
NUM_FRAMES  = 32
FLOW_PAIRS  = NUM_FRAMES - 1
BLUR_FRAMES = 8

def export_friendly_sdpa(query, key, value,
                          attn_mask=None, dropout_p=0.0,
                          is_causal=False, scale=None):
    scale_factor = scale if scale is not None else (1.0 / math.sqrt(query.size(-1)))
    attn_weight  = torch.matmul(query, key.transpose(-2, -1)) * scale_factor
    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_weight.masked_fill_(attn_mask.logical_not(), float("-inf"))
        else:
            attn_weight += attn_mask
    attn_weight = torch.softmax(attn_weight, dim=-1)
    return torch.matmul(attn_weight, value)

F.scaled_dot_product_attention = export_friendly_sdpa

try:
    import torch.onnx._internal.exporter._onnx_program
    torch.onnx._internal.exporter._onnx_program.ONNXProgram.optimize = (
        lambda self: self.model
    )
except Exception:
    pass

from fastvqa.models.evaluator import Stablev2Evaluator

sys.path.insert(0, "/workspace/StableVQA/NeuFlow")
from NeuFlow.neuflow import NeuFlow


def unwrap(m):
    return m.module if hasattr(m, "module") else m



class BackboneWrapper(nn.Module):
    """
    Input : [32, 3, 224, 224]  ImageNet-normalised frames (flat batch of T frames)
    Output: [1, 24576]         per-frame SwinV1 features concatenated temporally

    Mirrors evaluator.py:
        x        = vclips[key].reshape(-1, c, h, w)      # [n*d, 3, H, W]
        img_f    = resize_backbone(x)                    # [n*d, 768]
        img_feat = img_f.reshape(n, d * img_f.size(1))  # [1, 32*768] = [1, 24576]
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        feat = self.backbone(x)
        return feat.reshape(1, -1)


class NeuFlowWrapper(nn.Module):
    """
    Replaces RAFT. NeuFlow is fully batchable — all 31 pairs in one forward pass.

    Input : [31, 6, 224, 224]  31 frame-pairs [img1‖img2], raw [0,255] floats
    Output: [31, 2, 224, 224]  optical flow (u,v) for each pair

    NeuFlow must be pre-initialised with init_bhwd(31, 224, 224, device)
    before this wrapper is called.
    """
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        img1 = x[:, :3]
        img2 = x[:, 3:]
        out  = self.m(img1, img2)
        out  = out[-1] if isinstance(out, (list, tuple)) else out
        return out


class MotionWrapper(nn.Module):
    """
    Input : [1, 2, 32, 224, 224]
              — full 32-frame optical flow stack (31 real flows + 1 zero self-flow)
              — channel-0 = u-flow, channel-1 = v-flow

    This matches evaluator.py exactly:
        optical_feat = self.motion_analyzer(torch.stack(optical_flows, 2))
        # torch.stack(optical_flows, 2) stacks 32 flows along dim-2
        # → [1, 2, 32, 224, 224]

    Output: [1, 512]  3D-ResNet18 global-pooled feature
    """
    def __init__(self, m):
        super().__init__()
        self.m = unwrap(m)

    def forward(self, x):
        out = self.m(x)
        return out[0] if isinstance(out, (list, tuple)) else out


class DeblurWrapper(nn.Module):
    """
    Input : [8, 3, 224, 224]  ImageNet-normalised frames sampled at stride-4
    Output: [8, C, H', W']    Stripformer intermediate feature maps
    """
    def __init__(self, m):
        super().__init__()
        self.m = unwrap(m)

    def forward(self, x):
        return self.m(x)


class FusionHead(nn.Module):
    """
    Input : [1, 27648]
              Concatenation order MUST match evaluator.py:
                blur_feat    : 8  * 320 = 2560
                backbone_feat: 32 * 768 = 24576
                motion_feat  :            512
                total                  = 27648
    Output: [1, 1]  raw quality score
    """
    def __init__(self, m):
        super().__init__()
        self.m = unwrap(m)

    def forward(self, x):
        return self.m(x)


cfg   = yaml.safe_load(open(CONFIG_PATH))
model = Stablev2Evaluator(**cfg["model"].get("args", {}))

ckpt     = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
sd       = ckpt.get("state_dict") or ckpt.get("model") or ckpt
clean_sd = {}
for k, v in sd.items():
    if k.startswith("module."):
        k = k[len("module."):]
    if k.startswith("deblur_net.") and not k.startswith("deblur_net.module."):
        k = k.replace("deblur_net.", "deblur_net.module.", 1)
    clean_sd[k] = v

model.load_state_dict(clean_sd, strict=False)
model.eval()
model.cpu()

nf_ckpt  = torch.load(NEUFLOW_WEIGHTS, map_location="cpu", weights_only=False)
nf_sd    = nf_ckpt.get("model") or nf_ckpt.get("state_dict") or nf_ckpt
nf_clean = {k.replace("module.", ""): v for k, v in nf_sd.items()}

neuflow_gpu = NeuFlow()
neuflow_gpu.load_state_dict(nf_clean, strict=True)
neuflow_gpu.eval()
neuflow_gpu.to(DEVICE)
neuflow_gpu.init_bhwd(FLOW_PAIRS, TARGET_SIZE, TARGET_SIZE, str(DEVICE), amp=False)
flow_wrapper_gpu = NeuFlowWrapper(neuflow_gpu)
print(f"NeuFlow (GPU) initialised: batch={FLOW_PAIRS}, H={TARGET_SIZE}, W={TARGET_SIZE}")

neuflow_cpu = NeuFlow()
neuflow_cpu.load_state_dict(nf_clean, strict=True)
neuflow_cpu.eval()
neuflow_cpu.cpu()
neuflow_cpu.init_bhwd(FLOW_PAIRS, TARGET_SIZE, TARGET_SIZE, "cpu", amp=False)
flow_wrapper_cpu = NeuFlowWrapper(neuflow_cpu)

if os.path.exists(PREP_DIR):
    shutil.rmtree(PREP_DIR)
os.makedirs(PREP_DIR)

if os.path.exists(CALIB_DIR):
    shutil.rmtree(CALIB_DIR)

for b in ["backbone", "flow_model", "motion_analyzer", "deblur_net"]:
    os.makedirs(os.path.join(CALIB_DIR, b))


def export_and_preprocess(torch_model, dummy, name):
    raw      = os.path.join(PREP_DIR, f"{name}_raw.onnx")
    inferred = os.path.join(PREP_DIR, f"{name}_inferred.onnx")
    prep     = os.path.join(PREP_DIR, f"{name}_prep.onnx")

    torch.onnx.export(
        torch_model.eval(),
        dummy,
        raw,
        input_names=["input"],
        output_names=["output"],
        opset_version=18,
        dynamic_axes={
            "input":  {0: "batch"},
            "output": {0: "batch"},
        },
    )

    infer_shapes_path(raw, inferred)

    quant_pre_process(
        inferred,
        prep,
        skip_onnx_shape=True,
        skip_symbolic_shape=True,
        skip_optimization=True,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
    )

    for p in [raw, inferred, raw + ".data", inferred + ".data"]:
        if os.path.exists(p):
            os.remove(p)

    print(f"exported: {name}")



export_and_preprocess(
    flow_wrapper_cpu,
    torch.rand(FLOW_PAIRS, 6, TARGET_SIZE, TARGET_SIZE) * 255.0,
    "flow_model",
)

export_and_preprocess(
    BackboneWrapper(model.resize_backbone).cpu(),
    torch.randn(NUM_FRAMES, 3, TARGET_SIZE, TARGET_SIZE),
    "backbone",
)

export_and_preprocess(
    DeblurWrapper(unwrap(model.deblur_net)).cpu(),
    torch.randn(BLUR_FRAMES, 3, TARGET_SIZE, TARGET_SIZE),
    "deblur_net",
)

export_and_preprocess(
    MotionWrapper(model.motion_analyzer).cpu(),
    torch.randn(1, 2, NUM_FRAMES, TARGET_SIZE, TARGET_SIZE),
    "motion_analyzer",
)

input_size = 27648
for m in unwrap(model.quality).modules():
    if isinstance(m, nn.Linear):
        input_size = m.in_features
        break

qh = os.path.join(PREP_DIR, "quality_head_fp32.onnx")
torch.onnx.export(
    FusionHead(unwrap(model.quality)).eval(),
    torch.randn(1, input_size),
    qh,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    dynamic_axes={
        "input":  {0: "batch"},
        "output": {0: "batch"},
    },
)
print("exported: quality_head")


ORI_CLIP_LEN = NUM_FRAMES * 2

def preprocess_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total > ORI_CLIP_LEN:
        start = max(0, (total - ORI_CLIP_LEN + 1) // 2)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    norm_list, raw_list = [], []
    collected = 0
    i = 0

    while collected < NUM_FRAMES and cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if i % 2 == 0:
            frame    = cv2.resize(frame, (TARGET_SIZE, TARGET_SIZE))
            frame    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32)
            chw      = np.transpose(frame, (2, 0, 1))
            norm_chw = np.transpose((frame - MEAN) / STD, (2, 0, 1))
            raw_list.append(chw)
            norm_list.append(norm_chw)
            collected += 1
        i += 1

    cap.release()
    if len(norm_list) == 0:
        return None, None

    while len(norm_list) < NUM_FRAMES:
        norm_list.append(norm_list[-1])
        raw_list.append(raw_list[-1])

    return (
        torch.from_numpy(np.stack(norm_list)),
        torch.from_numpy(np.stack(raw_list)),
    )


videos = [f for f in os.listdir(VIDEO_FOLDER) if f.lower().endswith(".mp4")]
random.seed(RANDOM_SEED)
videos = random.sample(videos, min(NUM_CALIB_VIDS, len(videos)))

with torch.no_grad():
    for vid_idx, vid in enumerate(videos):

        motion_check = os.path.join(CALIB_DIR, "motion_analyzer", f"vid_{vid_idx}.pt")
        if os.path.exists(motion_check):
            print(f"{vid_idx + 1}/{len(videos)} already done, skipping")
            continue

        path = os.path.join(VIDEO_FOLDER, vid)
        norm_tensor, raw_tensor = preprocess_video(path)
        if norm_tensor is None:
            continue

        torch.save(norm_tensor,
                   os.path.join(CALIB_DIR, "backbone", f"vid_{vid_idx}.pt"))

        torch.save(norm_tensor[::4],
                   os.path.join(CALIB_DIR, "deblur_net", f"vid_{vid_idx}.pt"))

        pairs = torch.stack([
            torch.cat([raw_tensor[k], raw_tensor[k + 1]], dim=0)
            for k in range(FLOW_PAIRS)
        ])
        torch.save(pairs,
                   os.path.join(CALIB_DIR, "flow_model", f"vid_{vid_idx}.pt"))

        pairs_gpu = pairs.to(DEVICE)
        flow_out  = flow_wrapper_gpu(pairs_gpu)
        flow_out  = flow_out.cpu()

        all_flows = list(flow_out)
        all_flows.append(torch.zeros_like(flow_out[0]))

        motion = torch.stack(all_flows, dim=0).permute(1, 0, 2, 3).unsqueeze(0)
        torch.save(motion,
                   os.path.join(CALIB_DIR, "motion_analyzer", f"vid_{vid_idx}.pt"))

        print(f"{vid_idx + 1}/{len(videos)} done")

print("Calibration data generation complete.")

Using device: cuda
swinv1


/usr/local/lib/python3.10/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Setting backbone: resize_backbone
NeuFlow (GPU) initialised: batch=31, H=224, W=224


/tmp/ipykernel_3178197/93241413.py:58: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  scale_factor = scale if scale is not None else (1.0 / math.sqrt(query.size(-1)))
/workspace/StableVQA/NeuFlow/NeuFlow/corr.py:64: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  corr = corr.view(b*h*w, 1, h, w) / math.sqrt(c)


exported: flow_model


/workspace/StableVQA/fastvqa/models/swinv1_backbone.py:467: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert H == self.img_size[0] and W == self.img_size[1], \
/workspace/StableVQA/fastvqa/models/swinv1_backbone.py:251: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert L == H * W, "input feature has wrong size"
/workspace/StableVQA/fastvqa/models/swinv1_backbone.py:73: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a co

exported: backbone


/workspace/StableVQA/fastvqa/models/stripformer/Stripformer.py:232: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if H == W:
/workspace/StableVQA/fastvqa/models/stripformer/Stripformer.py:141: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  attention_head_size = int(C / self.num_attention_heads)
/workspace/StableVQA/fastvqa/models/stripformer/Stripformer.py:158: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the futu

exported: deblur_net
exported: motion_analyzer
exported: quality_head
1/200 done
2/200 done
3/200 done
4/200 done
5/200 done
6/200 done
7/200 done
8/200 done
9/200 done
10/200 done
11/200 done
12/200 done
13/200 done
14/200 done
15/200 done
16/200 done
17/200 done
18/200 done
19/200 done
20/200 done
21/200 done
22/200 done
23/200 done
24/200 done
25/200 done
26/200 done
27/200 done
28/200 done
29/200 done
30/200 done
31/200 done
32/200 done
33/200 done
34/200 done
35/200 done
36/200 done
37/200 done
38/200 done
39/200 done
40/200 done
41/200 done
42/200 done
43/200 done
44/200 done
45/200 done
46/200 done
47/200 done
48/200 done
49/200 done
50/200 done
51/200 done
52/200 done
53/200 done
54/200 done
55/200 done
56/200 done
57/200 done
58/200 done
59/200 done
60/200 done
61/200 done
62/200 done
63/200 done
64/200 done
65/200 done
66/200 done
67/200 done
68/200 done
69/200 done
70/200 done
71/200 done
72/200 done
73/200 done
74/200 done
75/200 done
76/200 done
77/200 done
78/200 done
79/

## Quantize exported models

In [1]:
import os
import shutil
import torch
import onnxruntime as ort

from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    CalibrationDataReader,
    CalibrationMethod,
)

PREP_DIR = "/workspace/StableVQA/NEWonnx_static_prep"
CALIB_DIR = "/workspace/StableVQA/NEWcalib_data"
OUT_DIR = "NEWonnx_static_int8"
WORK_DIR = "NEWonnx_static_work"

os.makedirs(OUT_DIR, exist_ok=True)

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

shutil.copytree(PREP_DIR, WORK_DIR)


class BranchCalibReader(CalibrationDataReader):
    def __init__(self, data_dir):
        self.files = sorted([
            f for f in os.listdir(data_dir)
            if f.endswith(".pt")
        ])[:200]

        self.data_dir = data_dir
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.files):
            return None

        f = self.files[self.idx]
        self.idx += 1

        tensor = torch.load(
            os.path.join(self.data_dir, f),
            weights_only=False
        )

        return {"input": tensor.numpy()}


BRANCHES = {
    "backbone": {
        "provider": ["CPUExecutionProvider"],
        "max_intermediate": 10,
    },
    "deblur_net": {
        "provider": ["CUDAExecutionProvider", "CPUExecutionProvider"],
        "max_intermediate": 4,
    },
    "flow_model": {
        "provider": ["CUDAExecutionProvider", "CPUExecutionProvider"],
        "max_intermediate": 4,
    },
    "motion_analyzer": {
        "provider": ["CUDAExecutionProvider", "CPUExecutionProvider"],
        "max_intermediate": 10,
    },
}


for branch, cfg in BRANCHES.items():
    prep = os.path.join(WORK_DIR, f"{branch}_prep.onnx")
    folded = os.path.join(WORK_DIR, f"{branch}_folded.onnx")
    out = os.path.join(OUT_DIR, f"{branch}_quant.onnx")
    calib = os.path.join(CALIB_DIR, branch)

    print("quantizing:", branch)

    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
    )
    sess_opts.optimized_model_filepath = folded

    ort.InferenceSession(
        prep,
        sess_opts,
        providers=["CPUExecutionProvider"]
    )

    reader = BranchCalibReader(calib)

    quantize_static(
        model_input=folded,
        model_output=out,
        calibration_data_reader=reader,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        calibrate_method=CalibrationMethod.MinMax,
        calibration_providers=cfg["provider"],
        use_external_data_format=False,
        extra_options={
            "WeightSymmetric": True,
            "MaxIntermediateOutputs": cfg["max_intermediate"],
        },
    )

    print("saved:", out)


src = os.path.join(WORK_DIR, "quality_head_fp32.onnx")
dst = os.path.join(OUT_DIR, "quality_head_quant.onnx")

if os.path.exists(src):
    shutil.copy(src, dst)

src_data = src + ".data"
dst_data = dst + ".data"

if os.path.exists(src_data):
    shutil.copy(src_data, dst_data)

print("copied: quality_head")
print("done")

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


quantizing: backbone
saved: NEWonnx_static_int8/backbone_quant.onnx
quantizing: deblur_net


/usr/local/lib/python3.10/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


saved: NEWonnx_static_int8/deblur_net_quant.onnx
quantizing: flow_model
saved: NEWonnx_static_int8/flow_model_quant.onnx
quantizing: motion_analyzer
saved: NEWonnx_static_int8/motion_analyzer_quant.onnx
copied: quality_head
done


## NEW

In [2]:
import os
import sys
import math

os.chdir("/workspace/StableVQA")
sys.path.insert(0, "/workspace/StableVQA")

import cv2
import yaml
import torch
import random
import shutil
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from onnx.shape_inference import infer_shapes_path
from onnxruntime.quantization import quant_pre_process

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

VIDEO_FOLDER    = "/workspace/VSdataset2.0"
CONFIG_PATH     = "/workspace/StableVQA/options/stable.yml"
WEIGHTS_PATH    = "/workspace/StableVQA/pretrained_weights/Stable-VQA-M-VS_val_n_dev_v1.0.pth"
NEUFLOW_WEIGHTS = "/workspace/StableVQA/NeuFlow/neuflow_mixed.pth"

NUM_CALIB_VIDS = 200
RANDOM_SEED    = 42

PREP_DIR  = "NEWonnx_static_prep"
CALIB_DIR = "NEWcalib_data"

MEAN = np.array([123.675, 116.28,  103.53], dtype=np.float32)
STD  = np.array([ 58.395,  57.12,  57.375], dtype=np.float32)

TARGET_SIZE = 224
NUM_FRAMES  = 32
FLOW_PAIRS  = NUM_FRAMES
BLUR_FRAMES = 8

def export_friendly_sdpa(query, key, value,
                          attn_mask=None, dropout_p=0.0,
                          is_causal=False, scale=None):
    scale_factor = scale if scale is not None else (1.0 / math.sqrt(query.size(-1)))
    attn_weight  = torch.matmul(query, key.transpose(-2, -1)) * scale_factor
    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_weight.masked_fill_(attn_mask.logical_not(), float("-inf"))
        else:
            attn_weight += attn_mask
    attn_weight = torch.softmax(attn_weight, dim=-1)
    return torch.matmul(attn_weight, value)

F.scaled_dot_product_attention = export_friendly_sdpa

try:
    import torch.onnx._internal.exporter._onnx_program
    torch.onnx._internal.exporter._onnx_program.ONNXProgram.optimize = (
        lambda self: self.model
    )
except Exception:
    pass

from fastvqa.models.evaluator import Stablev2Evaluator

sys.path.insert(0, "/workspace/StableVQA/NeuFlow")
from NeuFlow.neuflow import NeuFlow


def unwrap(m):
    return m.module if hasattr(m, "module") else m


def make_backbone_input_like_evaluator(frames):
    """frames: semantic [32,3,H,W] -> evaluator.py reshape input [32,3,H,W]."""
    d, c, h, w = frames.shape
    return frames.permute(1, 0, 2, 3).reshape(d, c, h, w)



class BackboneWrapper(nn.Module):
    """
    Input : [32, 3, 224, 224]  ImageNet-normalised evaluator-reshape frames
    Output: [1, 24576]         per-frame SwinV1 features concatenated temporally

    Mirrors evaluator.py:
        x        = vclips[key].reshape(-1, c, h, w)      # [n*d, 3, H, W]
        img_f    = resize_backbone(x)                    # [n*d, 768]
        img_feat = img_f.reshape(n, d * img_f.size(1))  # [1, 32*768] = [1, 24576]
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        feat = self.backbone(x)
        return feat.reshape(1, -1)


class NeuFlowWrapper(nn.Module):
    """
    Replaces RAFT. NeuFlow is fully batchable — all 32 pairs in one forward pass.

    Input : [32, 6, 224, 224]  32 normalized frame-pairs [img1‖img2]
    Output: [32, 2, 224, 224]  optical flow (u,v) for each pair

    NeuFlow must be pre-initialised with init_bhwd(32, 224, 224, device)
    before this wrapper is called.
    """
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        img1 = x[:, :3]
        img2 = x[:, 3:]
        out  = self.m(img1, img2)
        out  = out[-1] if isinstance(out, (list, tuple)) else out
        return out


class MotionWrapper(nn.Module):
    """
    Input : [1, 2, 32, 224, 224]
              — full 32-frame optical flow stack, including final self-flow
              — channel-0 = u-flow, channel-1 = v-flow

    This matches evaluator.py exactly:
        optical_feat = self.motion_analyzer(torch.stack(optical_flows, 2))
        # torch.stack(optical_flows, 2) stacks 32 flows along dim-2
        # → [1, 2, 32, 224, 224]

    Output: [1, 512]  3D-ResNet18 global-pooled feature
    """
    def __init__(self, m):
        super().__init__()
        self.m = unwrap(m)

    def forward(self, x):
        out = self.m(x)
        return out[0] if isinstance(out, (list, tuple)) else out


class DeblurWrapper(nn.Module):
    """
    Input : [8, 3, 224, 224]  ImageNet-normalised frames sampled at stride-4
    Output: [8, C, H', W']    Stripformer intermediate feature maps
    """
    def __init__(self, m):
        super().__init__()
        self.m = unwrap(m)

    def forward(self, x):
        return self.m(x)


class FusionHead(nn.Module):
    """
    Input : [1, 27648]
              Concatenation order MUST match evaluator.py:
                blur_feat    : 8  * 320 = 2560
                backbone_feat: 32 * 768 = 24576
                motion_feat  :            512
                total                  = 27648
    Output: [1, 1]  raw quality score
    """
    def __init__(self, m):
        super().__init__()
        self.m = unwrap(m)

    def forward(self, x):
        return self.m(x)


cfg   = yaml.safe_load(open(CONFIG_PATH))
model = Stablev2Evaluator(**cfg["model"].get("args", {}))

ckpt     = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
sd       = ckpt.get("state_dict") or ckpt.get("model") or ckpt
clean_sd = {}
for k, v in sd.items():
    if k.startswith("module."):
        k = k[len("module."):]
    if k.startswith("deblur_net.") and not k.startswith("deblur_net.module."):
        k = k.replace("deblur_net.", "deblur_net.module.", 1)
    clean_sd[k] = v

model.load_state_dict(clean_sd, strict=False)
model.eval()
model.cpu()

nf_ckpt  = torch.load(NEUFLOW_WEIGHTS, map_location="cpu", weights_only=False)
nf_sd    = nf_ckpt.get("model") or nf_ckpt.get("state_dict") or nf_ckpt
nf_clean = {k.replace("module.", ""): v for k, v in nf_sd.items()}

neuflow_gpu = NeuFlow()
neuflow_gpu.load_state_dict(nf_clean, strict=True)
neuflow_gpu.eval()
neuflow_gpu.to(DEVICE)
neuflow_gpu.init_bhwd(FLOW_PAIRS, TARGET_SIZE, TARGET_SIZE, str(DEVICE), amp=False)
flow_wrapper_gpu = NeuFlowWrapper(neuflow_gpu)
print(f"NeuFlow (GPU) initialised: batch={FLOW_PAIRS}, H={TARGET_SIZE}, W={TARGET_SIZE}")

neuflow_cpu = NeuFlow()
neuflow_cpu.load_state_dict(nf_clean, strict=True)
neuflow_cpu.eval()
neuflow_cpu.cpu()
neuflow_cpu.init_bhwd(FLOW_PAIRS, TARGET_SIZE, TARGET_SIZE, "cpu", amp=False)
flow_wrapper_cpu = NeuFlowWrapper(neuflow_cpu)

if os.path.exists(PREP_DIR):
    shutil.rmtree(PREP_DIR)
os.makedirs(PREP_DIR)

if os.path.exists(CALIB_DIR):
    shutil.rmtree(CALIB_DIR)

for b in ["backbone", "flow_model", "motion_analyzer", "deblur_net"]:
    os.makedirs(os.path.join(CALIB_DIR, b))


def export_and_preprocess(torch_model, dummy, name):
    raw      = os.path.join(PREP_DIR, f"{name}_raw.onnx")
    inferred = os.path.join(PREP_DIR, f"{name}_inferred.onnx")
    prep     = os.path.join(PREP_DIR, f"{name}_prep.onnx")

    torch.onnx.export(
        torch_model.eval(),
        dummy,
        raw,
        input_names=["input"],
        output_names=["output"],
        opset_version=18,
        dynamic_axes={
            "input":  {0: "batch"},
            "output": {0: "batch"},
        },
    )

    infer_shapes_path(raw, inferred)

    quant_pre_process(
        inferred,
        prep,
        skip_onnx_shape=True,
        skip_symbolic_shape=True,
        skip_optimization=True,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
    )

    for p in [raw, inferred, raw + ".data", inferred + ".data"]:
        if os.path.exists(p):
            os.remove(p)

    print(f"exported: {name}")



export_and_preprocess(
    flow_wrapper_cpu,
    torch.randn(FLOW_PAIRS, 6, TARGET_SIZE, TARGET_SIZE),
    "flow_model",
)

export_and_preprocess(
    BackboneWrapper(model.resize_backbone).cpu(),
    torch.randn(NUM_FRAMES, 3, TARGET_SIZE, TARGET_SIZE),
    "backbone",
)

export_and_preprocess(
    DeblurWrapper(unwrap(model.deblur_net)).cpu(),
    torch.randn(BLUR_FRAMES, 3, TARGET_SIZE, TARGET_SIZE),
    "deblur_net",
)

export_and_preprocess(
    MotionWrapper(model.motion_analyzer).cpu(),
    torch.randn(1, 2, NUM_FRAMES, TARGET_SIZE, TARGET_SIZE),
    "motion_analyzer",
)

input_size = 27648
for m in unwrap(model.quality).modules():
    if isinstance(m, nn.Linear):
        input_size = m.in_features
        break

qh = os.path.join(PREP_DIR, "quality_head_fp32.onnx")
torch.onnx.export(
    FusionHead(unwrap(model.quality)).eval(),
    torch.randn(1, input_size),
    qh,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    dynamic_axes={
        "input":  {0: "batch"},
        "output": {0: "batch"},
    },
)
print("exported: quality_head")


ORI_CLIP_LEN = NUM_FRAMES * 2

def preprocess_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total > ORI_CLIP_LEN:
        start = max(0, (total - ORI_CLIP_LEN + 1) // 2)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    norm_list, raw_list = [], []
    collected = 0
    i = 0

    while collected < NUM_FRAMES and cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if i % 2 == 0:
            frame    = cv2.resize(frame, (TARGET_SIZE, TARGET_SIZE))
            frame    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32)
            chw      = np.transpose(frame, (2, 0, 1))
            norm_chw = np.transpose((frame - MEAN) / STD, (2, 0, 1))
            raw_list.append(chw)
            norm_list.append(norm_chw)
            collected += 1
        i += 1

    cap.release()
    if len(norm_list) == 0:
        return None, None

    while len(norm_list) < NUM_FRAMES:
        norm_list.append(norm_list[-1])
        raw_list.append(raw_list[-1])

    return (
        torch.from_numpy(np.stack(norm_list)),
        torch.from_numpy(np.stack(raw_list)),
    )


videos = [f for f in os.listdir(VIDEO_FOLDER) if f.lower().endswith(".mp4")]
random.seed(RANDOM_SEED)
videos = random.sample(videos, min(NUM_CALIB_VIDS, len(videos)))

with torch.no_grad():
    for vid_idx, vid in enumerate(videos):

        motion_check = os.path.join(CALIB_DIR, "motion_analyzer", f"vid_{vid_idx}.pt")
        if os.path.exists(motion_check):
            print(f"{vid_idx + 1}/{len(videos)} already done, skipping")
            continue

        path = os.path.join(VIDEO_FOLDER, vid)
        norm_tensor, raw_tensor = preprocess_video(path)
        if norm_tensor is None:
            continue

        torch.save(make_backbone_input_like_evaluator(norm_tensor),
                   os.path.join(CALIB_DIR, "backbone", f"vid_{vid_idx}.pt"))

        torch.save(norm_tensor[::4],
                   os.path.join(CALIB_DIR, "deblur_net", f"vid_{vid_idx}.pt"))

        pairs = torch.stack([
            torch.cat([norm_tensor[k], norm_tensor[min(k + 1, NUM_FRAMES - 1)]], dim=0)
            for k in range(FLOW_PAIRS)
        ])
        torch.save(pairs,
                   os.path.join(CALIB_DIR, "flow_model", f"vid_{vid_idx}.pt"))

        pairs_gpu = pairs.to(DEVICE)
        flow_out  = flow_wrapper_gpu(pairs_gpu)
        flow_out  = flow_out.cpu()

        motion = flow_out.permute(1, 0, 2, 3).unsqueeze(0)
        torch.save(motion,
                   os.path.join(CALIB_DIR, "motion_analyzer", f"vid_{vid_idx}.pt"))

        print(f"{vid_idx + 1}/{len(videos)} done")

print("Calibration data generation complete.")

Using device: cuda
swinv1


/usr/local/lib/python3.10/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Setting backbone: resize_backbone
NeuFlow (GPU) initialised: batch=32, H=224, W=224


/tmp/ipykernel_3254590/4170432873.py:58: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  scale_factor = scale if scale is not None else (1.0 / math.sqrt(query.size(-1)))
/workspace/StableVQA/NeuFlow/NeuFlow/corr.py:64: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  corr = corr.view(b*h*w, 1, h, w) / math.sqrt(c)


exported: flow_model


/workspace/StableVQA/fastvqa/models/swinv1_backbone.py:467: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert H == self.img_size[0] and W == self.img_size[1], \
/workspace/StableVQA/fastvqa/models/swinv1_backbone.py:251: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert L == H * W, "input feature has wrong size"
/workspace/StableVQA/fastvqa/models/swinv1_backbone.py:73: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a co

exported: backbone


/workspace/StableVQA/fastvqa/models/stripformer/Stripformer.py:232: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if H == W:
/workspace/StableVQA/fastvqa/models/stripformer/Stripformer.py:141: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  attention_head_size = int(C / self.num_attention_heads)
/workspace/StableVQA/fastvqa/models/stripformer/Stripformer.py:158: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the futu

exported: deblur_net
exported: motion_analyzer
exported: quality_head
1/200 done
2/200 done
3/200 done
4/200 done
5/200 done
6/200 done
7/200 done
8/200 done
9/200 done
10/200 done
11/200 done
12/200 done
13/200 done
14/200 done
15/200 done
16/200 done
17/200 done
18/200 done
19/200 done
20/200 done
21/200 done
22/200 done
23/200 done
24/200 done
25/200 done
26/200 done
27/200 done
28/200 done
29/200 done
30/200 done
31/200 done
32/200 done
33/200 done
34/200 done
35/200 done
36/200 done
37/200 done
38/200 done
39/200 done
40/200 done
41/200 done
42/200 done
43/200 done
44/200 done
45/200 done
46/200 done
47/200 done
48/200 done
49/200 done
50/200 done
51/200 done
52/200 done
53/200 done
54/200 done
55/200 done
56/200 done
57/200 done
58/200 done
59/200 done
60/200 done
61/200 done
62/200 done
63/200 done
64/200 done
65/200 done
66/200 done
67/200 done
68/200 done
69/200 done
70/200 done
71/200 done
72/200 done
73/200 done
74/200 done
75/200 done
76/200 done
77/200 done
78/200 done
79/

In [1]:
import os
import shutil
import torch
import onnxruntime as ort

from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    CalibrationDataReader,
    CalibrationMethod,
)

PREP_DIR = "/workspace/StableVQA/NEWonnx_static_prep"
CALIB_DIR = "/workspace/StableVQA/NEWcalib_data"
OUT_DIR = "NEWonnx_static_int8"
WORK_DIR = "NEWonnx_static_work"

os.makedirs(OUT_DIR, exist_ok=True)

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

shutil.copytree(PREP_DIR, WORK_DIR)


class BranchCalibReader(CalibrationDataReader):
    def __init__(self, data_dir):
        self.files = sorted([
            f for f in os.listdir(data_dir)
            if f.endswith(".pt")
        ])[:200]

        self.data_dir = data_dir
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.files):
            return None

        f = self.files[self.idx]
        self.idx += 1

        tensor = torch.load(
            os.path.join(self.data_dir, f),
            weights_only=False
        )

        return {"input": tensor.numpy()}


BRANCHES = {
    "backbone": {
        "provider": ["CPUExecutionProvider"],
        "max_intermediate": 10,
    },
    "deblur_net": {
        "provider": ["CUDAExecutionProvider", "CPUExecutionProvider"],
        "max_intermediate": 4,
    },
    "flow_model": {
        "provider": ["CUDAExecutionProvider", "CPUExecutionProvider"],
        "max_intermediate": 4,
    },
    "motion_analyzer": {
        "provider": ["CUDAExecutionProvider", "CPUExecutionProvider"],
        "max_intermediate": 10,
    },
}


for branch, cfg in BRANCHES.items():
    prep = os.path.join(WORK_DIR, f"{branch}_prep.onnx")
    folded = os.path.join(WORK_DIR, f"{branch}_folded.onnx")
    out = os.path.join(OUT_DIR, f"{branch}_quant.onnx")
    calib = os.path.join(CALIB_DIR, branch)

    print("quantizing:", branch)

    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
    )
    sess_opts.optimized_model_filepath = folded

    ort.InferenceSession(
        prep,
        sess_opts,
        providers=["CPUExecutionProvider"]
    )

    reader = BranchCalibReader(calib)

    quantize_static(
        model_input=folded,
        model_output=out,
        calibration_data_reader=reader,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        calibrate_method=CalibrationMethod.MinMax,
        calibration_providers=cfg["provider"],
        use_external_data_format=False,
        extra_options={
            "WeightSymmetric": True,
            "MaxIntermediateOutputs": cfg["max_intermediate"],
        },
    )

    print("saved:", out)


src = os.path.join(WORK_DIR, "quality_head_fp32.onnx")
dst = os.path.join(OUT_DIR, "quality_head_quant.onnx")

if os.path.exists(src):
    shutil.copy(src, dst)

src_data = src + ".data"
dst_data = dst + ".data"

if os.path.exists(src_data):
    shutil.copy(src_data, dst_data)

print("copied: quality_head")
print("done")

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


quantizing: backbone
saved: NEWonnx_static_int8/backbone_quant.onnx
quantizing: deblur_net


/usr/local/lib/python3.10/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


saved: NEWonnx_static_int8/deblur_net_quant.onnx
quantizing: flow_model
saved: NEWonnx_static_int8/flow_model_quant.onnx
quantizing: motion_analyzer
saved: NEWonnx_static_int8/motion_analyzer_quant.onnx
copied: quality_head
done
